# D04 · I2C on the wire - conditions, pull-ups, recovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D04_i2c_on_the_wire_conditions_pull_ups_recovery.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 55 min

**Skill Hub concepts:** I2C start, stop, repeated start and acknowledge; I2C bus recovery; Clock stretching; I2C pull-up sizing

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D04")

## Map: two open-drain wires, four conditions

| Condition | SCL | SDA | Meaning |
|---|---|---|---|
| START (S) | high | falls | bus taken |
| Repeated START (Sr) | high | falls, without a STOP before it | switch to read without releasing the bus |
| STOP (P) | high | rises | bus released |
| Data bit | low while SDA changes, **sampled on SCL rising** | stable while SCL high | |
| ACK / NACK | 9th clock | receiver pulls low = ACK, leaves high = NACK | |
| Clock stretching | a slave holds SCL low | | "not ready yet"; master must wait until SCL really reads high |

Register read: `S | addr<<1|0 A | reg A | Sr | addr<<1|1 A | data NACK | P`

Pull-up sizing (rise time from 30% to 70%): $t_r = 0.8473\,R_p\,C_b$
$$R_{min} = \frac{V_{DD} - V_{OL}}{I_{OL}} \qquad R_{max} = \frac{t_r^{max}}{0.8473\,C_b}$$

| Mode | Clock | $t_r^{max}$ | $I_{OL}$ at $V_{OL}=0.4$ V |
|---|---|---|---|
| Standard | 100 kHz | 1000 ns | 3 mA |
| Fast | 400 kHz | 300 ns | 3 mA |
| Fast-mode Plus | 1 MHz | 120 ns | 20 mA |

Bus recovery: if a reset hits the master mid-read, the slave may still hold SDA low waiting for clocks. Toggle SCL
(up to 9 times) until SDA reads high, then send a STOP.

## Task 1: decode a capture into events
Return a list of `("S",)`, `("Sr",)`, `("P",)` and `("B", value, acked)` tuples. A START seen while a transaction is
open (no STOP since the last START) is `"Sr"`.

In [ ]:
def i2c_decode(scl, sda):
    raise NotImplementedError  # TODO

lab.check("decode", i2c_decode)

In [ ]:
%%writefile d04_decode.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/d04.h
// Grade it with lab.check("decode", "d04_decode.c"). For C++, name the file .cpp in both places.
#include "d04.h"

size_t i2c_decode(const uint8_t *scl, const uint8_t *sda, size_t n, I2cEvent *out, size_t out_cap) {
    (void)scl; (void)sda; (void)n; (void)out; (void)out_cap;
    return 0; /* TODO: START/STOP = SDA edge while SCL high; data = sample SDA on rising SCL, 8 bits + ACK */
}

In [ ]:
# lab.check("decode", "d04_decode.c")  # uncomment after writing the C/C++ cell above

## Task 2: pull-up range

In [ ]:
def pullup_range(vdd, cb_pF, mode):
    """mode in {"standard", "fast", "fastplus"}; return (r_min_ohm, r_max_ohm)."""
    raise NotImplementedError  # TODO

lab.check("pullup", pullup_range)

In [ ]:
%%writefile d04_pullup.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/d04.h
// Grade it with lab.check("pullup", "d04_pullup.c"). For C++, name the file .cpp in both places.
#include "d04.h"

void pullup_range(double vdd, double cb_pF, int mode, double *r_min_ohm, double *r_max_ohm) {
    (void)vdd; (void)cb_pF; (void)mode;
    *r_min_ohm = 0; *r_max_ohm = 0; /* TODO */
}

In [ ]:
# lab.check("pullup", "d04_pullup.c")  # uncomment after writing the C/C++ cell above

## Task 3: recover a stuck bus
`bus.sda()` reads SDA, `bus.pulse_scl()` gives one clock, `bus.stop()` sends a STOP (only valid while SDA is high).
Return how many pulses you used. Never send more than 9.

In [ ]:
def recover(bus):
    raise NotImplementedError  # TODO

lab.check("recover", recover)

In [ ]:
%%writefile d04_recover.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/d04.h
// Grade it with lab.check("recover", "d04_recover.c"). For C++, name the file .cpp in both places.
#include "d04.h"

int i2c_recover(const I2cBusOps *bus) {
    (void)bus;
    return 0; /* TODO: pulse SCL while SDA is low (at most 9 times), then STOP */
}

In [ ]:
# lab.check("recover", "d04_recover.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** $R_{min}$ in ohms for 3.3 V, standard mode.

**M2 (Apply).** $R_{max}$ in ohms for fast mode with 200 pF of bus capacitance.

**M3 (Apply).** Address byte that starts a read from the 7-bit device `0x68` (an IMU)?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** During clock stretching, who drives what?
- a) the master holds SDA low until the slave is ready
- b) the slave holds SCL low; the master releases SCL and waits until it actually reads high before continuing
- c) both devices stop driving and the pull-up times out

**K2 (Analyze).** After a watchdog reset the I2C bus reads SDA = 0, SCL = 1 forever. Explain why, why a STOP alone
cannot fix it, and the recovery sequence you put in your init code.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d04.h`).
3. In `roboprep/checks/d04.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()